# 02 · 事件流 & 流式输出

目标：掌握 SDK 的事件模型，能把 delta 流封装成 async generator 给上层（SSE / WebSocket）消费。

## 0. 事件大图 —— 一次 send 都会触发哪些事件？

SDK 的**唯一**输出通道就是事件流。`session.send` 只是个 "fire" 动作，所有反馈都通过 `session.on(handler)` 推回来。下面这张图把一轮典型对话的事件按时序画出来：

```mermaid
%%{init: {'theme':'base','themeVariables':{
  'primaryColor':'#1e293b','primaryTextColor':'#f8fafc','primaryBorderColor':'#475569',
  'lineColor':'#94a3b8','secondaryColor':'#334155','tertiaryColor':'#1e293b',
  'actorBkg':'#334155','actorTextColor':'#f8fafc','actorBorder':'#64748b',
  'signalColor':'#cbd5e1','signalTextColor':'#f1f5f9',
  'noteBkgColor':'#fef3c7','noteTextColor':'#1e293b','noteBorderColor':'#d97706'
}}}%%
sequenceDiagram
    autonumber
    participant App as 业务代码
    participant Sess as CopilotSession
    participant CLI as copilot CLI

    App->>Sess: session.send("prompt")
    CLI-->>Sess: user.message
    rect rgb(30, 58, 95)
        Note over CLI,Sess: 模型生成回复（streaming=True 才有 delta）
        CLI-->>Sess: assistant.reasoning.delta · · ·
        CLI-->>Sess: assistant.reasoning
        CLI-->>Sess: assistant.message.delta · · ·
        CLI-->>Sess: assistant.message  ← 总是有
    end
    rect rgb(120, 53, 15)
        Note over CLI,Sess: 模型决定调工具（可能多次循环）
        CLI-->>Sess: tool.executionStart
        Note right of CLI: 未传 on_permission_request<br/>会发 permission.requested
        CLI-->>Sess: tool.executionComplete
        CLI-->>Sess: assistant.message.delta · · ·
        CLI-->>Sess: assistant.message  ← 最终答复
    end
    rect rgb(20, 83, 45)
        CLI-->>Sess: session.idle  ← 本轮结束信号
    end
```

**事件分类心智模型**：

```mermaid
%%{init: {'theme':'base','themeVariables':{
  'primaryColor':'#1e293b','primaryTextColor':'#f8fafc','primaryBorderColor':'#475569',
  'lineColor':'#94a3b8','fontSize':'14px'
}}}%%
flowchart TD
    Event["SessionEvent<br/>(event.type + event.data)"]:::root
    Event --> User["<b>user.*</b><br/>UserMessageData"]:::user
    Event --> Assistant["<b>assistant.*</b><br/>消息 / reasoning / 增量"]:::asst
    Event --> Tool["<b>tool.*</b><br/>start / complete"]:::tool
    Event --> Perm["<b>permission.*</b><br/>未传 handler 时"]:::perm
    Event --> SessLC["<b>session.*</b><br/>idle / error / compaction"]:::lifecycle
    Event --> Ext["<b>external_tool.requested</b><br/>handler=None 时"]:::tool
    Event --> SubAgent["<b>subagent.*</b><br/>custom_agents 委派时"]:::asst

    Assistant --> A1["AssistantMessageData<br/>完整消息（一定有）"]:::asst
    Assistant --> A2["AssistantMessageDeltaData<br/>逐 token（仅 streaming=True）"]:::asst
    Assistant --> A3["AssistantReasoningData<br/>思考链（o-系列 / Claude）"]:::asst

    classDef root fill:#0f172a,stroke:#facc15,stroke-width:2px,color:#fef3c7
    classDef user fill:#0c4a6e,stroke:#38bdf8,color:#e0f2fe
    classDef asst fill:#581c87,stroke:#c084fc,color:#f3e8ff
    classDef tool fill:#14532d,stroke:#4ade80,color:#dcfce7
    classDef perm fill:#7f1d1d,stroke:#f87171,color:#fee2e2
    classDef lifecycle fill:#713f12,stroke:#facc15,color:#fef3c7
```

> **关键 takeaway**：等 `SessionIdleData` 才算一轮完成。不要靠 "看到 `AssistantMessageData` 就 return"，因为模型可能先输出一段、再调工具、再输出最终回复——这时**至少**会有 2 条 `AssistantMessageData`。


## 1. 事件订阅

```python
unsubscribe = session.on(handler)   # handler(event) - 同步函数
unsubscribe()                       # 取消订阅
```

`event` 主要字段：
- `event.type` —— 字符串，如 `'assistant.message'`、`'tool.executionStart'`
- `event.data` —— 结构化数据；推荐用 `match event.data: case AssistantMessageData(): ...` 做类型分支
- `event.sessionId`

客户端层也有 `client.on(...)` 用来订阅 **生命周期** 事件
（`session.created` / `session.deleted` / `session.foreground` / `session.background`）。

## 2. 常见事件类型对照

| 事件 type | data class | 何时触发 |
|---|---|---|
| `user.message` | `UserMessageData` | 你 `send()` 之后 |
| `assistant.message.delta` | `AssistantMessageDeltaData` | 仅在 `streaming=True` 时；逐 token |
| `assistant.message` | `AssistantMessageData` | 完整回复（**总是**会发，无论 streaming） |
| `assistant.reasoning.delta` / `assistant.reasoning` | `AssistantReasoningDeltaData` / `AssistantReasoningData` | 支持 reasoning 的模型 |
| `tool.executionStart` / `tool.executionComplete` | — | 工具调用前后 |
| `permission.requested` / `permission.completed` | `PermissionRequest` / ... | 当未传 `on_permission_request` 时 |
| `session.compaction_start` / `_complete` | — | infinite sessions 自动压缩 |
| `session.idle` | `SessionIdleData` | **本轮结束**，可解锁等待 |
| `session.error` | `SessionErrorData` | 运行时错误 |

## 3. 启用 streaming

In [ ]:
import os, asyncio
from dotenv import load_dotenv
from copilot import CopilotClient
from copilot.session import PermissionHandler
from copilot.generated.session_events import (
    AssistantMessageData, AssistantMessageDeltaData, SessionIdleData,
)

# 加载 .env 获取 GPT 5.4 端点
load_dotenv()

azure_provider = {
    'type': 'azure',
    'base_url': os.environ['AZURE_OPENAI_ENDPOINT_GPT_5_4'],
    'api_key': os.environ['AZURE_OPENAI_API_KEY_GPT_5_4'],
    'azure': {'api_version': os.getenv('AZURE_OPENAI_API_VERSION_GPT_5_4', '2025-03-01-preview')},
}

async def stream_demo(prompt: str):
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4'),
            provider=azure_provider,
            streaming=True,
        ) as session:
            done = asyncio.Event()
            def on_event(event):
                match event.data:
                    case AssistantMessageDeltaData() as d:
                        print(d.delta_content or '', end='', flush=True)
                    case AssistantMessageData() as d:
                        print(f'\n[final len={len(d.content)}]')
                    case SessionIdleData():
                        done.set()
            session.on(on_event)
            await session.send(prompt)
            await done.wait()

await stream_demo('用 3 句话讲一个关于章鱼工程师的小故事。')

### 3.1 对比：关掉 streaming

`streaming=False` 时不会有任何 `*.delta` 事件，但 **一定** 会在终态给一条完整的 `AssistantMessageData`。所以业务代码若只关心"最终答复"，可以完全忽略 delta 分支。


In [ ]:
async def non_stream_demo(prompt: str):
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4'),
            provider=azure_provider,
            streaming=False,   # ← 关闭流式
        ) as session:
            done = asyncio.Event()
            final = {'text': ''}
            seen_delta = {'n': 0}
            def on_event(event):
                match event.data:
                    case AssistantMessageDeltaData():
                        seen_delta['n'] += 1   # 不应该出现
                    case AssistantMessageData() as d:
                        final['text'] = d.content
                    case SessionIdleData():
                        done.set()
            session.on(on_event)
            await session.send(prompt)
            await done.wait()
            print(f'delta 事件数 = {seen_delta["n"]} (应为 0)')
            print(f'final: {final["text"][:120]}...')

await non_stream_demo('一句话说明 Python 的 GIL 是什么。')


## 4. 封装成 async generator（适合 FastAPI SSE）

SDK 把数据**推**给你（callback），但 FastAPI / WebSocket / SSE 通常需要**拉**（async iterator）。中间架一个 `asyncio.Queue` 当桥梁，再用 `None` 作哨兵代表「本轮完成」：

```mermaid
flowchart LR
    subgraph CLI["copilot CLI"]
        E1[delta] --> E2[delta] --> E3[delta] --> Idle[session.idle]
    end
    subgraph Bridge["stream_text() 回调"]
        OnEvent["on_event(event)"]
        Queue[("asyncio.Queue")]
        OnEvent -- "delta_content" --> Queue
        OnEvent -- "idle → put_nowait(None)" --> Queue
    end
    subgraph Consumer["业务侧 async for"]
        Loop["while item := await queue.get():<br/>    yield item"]
        Out[(FastAPI SSE / WS / stdout)]
        Loop --> Out
    end
    CLI -- session.on(handler) --> OnEvent
    Queue --> Loop
```

模式好处：
- 业务侧拿到的是干净的 `AsyncGenerator[str, None]`，无需懂 SDK 内部事件类型
- 错误也通过 queue 传递（见下方 `SessionErrorData` 分支）
- 多个消费者可以共享同一个 queue 或各自一份（fan-out 时把 handler 内的 `queue` 列表化即可）


In [ ]:
import asyncio
from typing import AsyncGenerator
from copilot.generated.session_events import (
    AssistantMessageDeltaData, SessionIdleData, SessionErrorData,
)

async def stream_text(session, prompt: str) -> AsyncGenerator[str, None]:
    queue: asyncio.Queue[str | None] = asyncio.Queue()
    def on_event(event):
        match event.data:
            case AssistantMessageDeltaData() as d:
                if d.delta_content:
                    queue.put_nowait(d.delta_content)
            case SessionErrorData() as d:
                queue.put_nowait(f'[error] {d.message}')
                queue.put_nowait(None)
            case SessionIdleData():
                queue.put_nowait(None)  # 终止信号
    unsubscribe = session.on(on_event)
    try:
        await session.send(prompt)
        while True:
            item = await queue.get()
            if item is None:
                return
            yield item
    finally:
        unsubscribe()

In [ ]:
async def run_text_stream():
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4'),
            provider=azure_provider,
            streaming=True,
        ) as session:
            async for token in stream_text(session, '用一句话讲一个关于海星的冷笑话。'):
                print(token, end='', flush=True)

await run_text_stream()

### 4.1 一次生成、多路消费（fan-out）

把单个 `asyncio.Queue` 换成 **多个 queue 的列表**，handler 里 `for q in queues: q.put_nowait(...)`，就能同时喂给 SSE 客户端、日志、监控等多个消费者。下面演示同一轮回复被两个 `async for` 同时消费：


In [ ]:
from typing import AsyncGenerator

async def stream_text_fanout(session, prompt: str, n: int = 2):
    queues = [asyncio.Queue() for _ in range(n)]
    def on_event(event):
        match event.data:
            case AssistantMessageDeltaData() as d:
                if d.delta_content:
                    for q in queues:
                        q.put_nowait(d.delta_content)
            case SessionIdleData():
                for q in queues:
                    q.put_nowait(None)
    unsub = session.on(on_event)

    async def consumer(idx: int, q: asyncio.Queue):
        chars = 0
        while (item := await q.get()) is not None:
            chars += len(item)
        print(f'[consumer-{idx}] 收到 {chars} 字符')

    await session.send(prompt)
    try:
        await asyncio.gather(*(consumer(i, q) for i, q in enumerate(queues)))
    finally:
        unsub()

async def run_fanout():
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4'),
            provider=azure_provider,
            streaming=True,
        ) as session:
            await stream_text_fanout(session, '列出 3 条 async 编程的最佳实践。', n=3)

await run_fanout()


## 5. 历史与 abort

- `await session.get_messages()` —— 拉取所有历史事件（用于持久化 / 调试）
- `await session.abort()` —— 中断当前正在跑的轮次（不销毁 session）
- `session.workspace_path` —— 当 `infinite_sessions` 开启时，CLI 为该 session 维护的目录 `~/.copilot/session-state/{session_id}/`，可读写文件


### 5.1 演示：发送中途 abort，并查看 `get_messages()`

模拟用户按下 "停止" —— `await session.abort()` 中断当前轮次，但 session 本身仍可继续 send。abort 之后通常会立即收到一条 `SessionIdleData` 把这一轮关掉。


In [ ]:
from copilot.generated.session_events import SessionErrorData

async def abort_demo():
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4'),
            provider=azure_provider,
            streaming=True,
        ) as session:
            done = asyncio.Event()
            chars = {'n': 0}
            def on_event(event):
                match event.data:
                    case AssistantMessageDeltaData() as d:
                        chars['n'] += len(d.delta_content or '')
                    case SessionIdleData():
                        done.set()
            session.on(on_event)

            await session.send('用 500 字详细解释 TCP 三次握手。')
            await asyncio.sleep(1.5)              # 跑一会儿
            print(f'abort 前已收到 {chars["n"]} 字符 → 触发 abort')
            await session.abort()
            await asyncio.wait_for(done.wait(), timeout=10)

            # session 还活着，可以查历史
            history = await session.get_messages()
            print(f'session.get_messages() 返回 {len(history)} 条事件')
            print(f'最后一条 type = {history[-1].type}')

await abort_demo()


## 6. Takeaways

- 等 `SessionIdleData` 是 **唯一可靠**的一轮结束信号
- streaming 关闭时也会有最终 `assistant.message`
- 用 Queue + idle 哨兵把回调式 API 封装成 async generator，几乎所有上层框架都能直接 pipe


## 7. 下一步

| Notebook | 主题 |
|---|---|
| [03_custom_tools.ipynb](03_custom_tools.ipynb) | `@define_tool` / `tools=[...]` 注入自定义工具 |
| [04_permissions_and_hooks.ipynb](04_permissions_and_hooks.ipynb) | `on_permission_request` 细粒度授权、生命周期 hooks |
| [05_byok_and_providers.ipynb](05_byok_and_providers.ipynb) | Azure / OpenAI / Anthropic provider 切换 |
